In [1]:
pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pdfplumber
import re

In [4]:
pdf_path = "/Users/abhishek/Desktop/world-bank-indicators-project/National_Classification_of_Occupations_Vol_II-B-2015.pdf"

In [11]:
with pdfplumber.open(pdf_path) as pdf:
    # Let's look at a page from the middle to see how the text is structured
    # For example, page 31 (index 30) 
    page = pdf.pages[30] 
    text = page.extract_text()
    
    # Print the first 1000 characters to see what the raw extraction looks like
    print(text[:1000])

National Classification of Occupations – 2015 Divison 6
vacuum. Ties hind legs of animals and Stableman
washes hind quarter with chlorinated
water. Cleans udder with plain chlorinated Stableman, Syce grooms, feeds and looks
water and messages them by hand. after horses, camels, mules etc., in
Disinfects hand by washing with stables. Grooms animals by brushing,
chlorinated water. Adjusts buckets currying, washing them and trimming
between own legs below udder. Presses their manes and tails using hair and metal
udders alternatively with fingers to extract brushes, clippers and scissors. Waters
milk from them into bucket. Stimulates feeds and exercises animals. Sweeps out
flow of milk from udder by repeated stalls and spreads fresh straw. May treat
supply of feed. Repeats pressing of udders sick or injured animal according to his
to extract maximum milk. Carries bucket knowledge or may report their condition
containing milk to specified place of to Veterinarian and follow his medical
stor

In [12]:
# Create an empty list to hold the text from each page
all_pages_text = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text: # If the page has text
            # Replace newlines with spaces so it's one long string per page
            clean_text = text.replace('\n', ' ')
            all_pages_text.append(clean_text)

print(f"Extracted text from {len(all_pages_text)} pages.")

Extracted text from 577 pages.


In [13]:
# 1. Define our keyword dictionaries
routine_words = ['machine', 'operate', 'assemble', 'repeat', 'monitor', 'record', 'calculate', 'check', 'sort', 'file']
nonroutine_words = ['repair', 'manage', 'design', 'teach', 'persuade', 'negotiate', 'advise', 'plan', 'organize', 'supervise']

# 2. Join all pages into one giant string
full_text = " ".join(all_pages_text)

# 3. Find all 4-digit codes and the text that follows them
# Regex looks for 4 digits, then grabs up to 300 characters after it
matches = re.findall(r'\b(\d{4})\b(.{1,300})', full_text)

# 4. Create an empty list to hold our rows
data_rows = []

# 5. Loop through the matches
for code, description in matches:
    if code == '2015': # Skip the publication year
        continue
        
    desc_lower = description.lower()
    
    # Count how many routine and non-routine words are in this description
    r_score = sum([1 for word in routine_words if word in desc_lower])
    nr_score = sum([1 for word in nonroutine_words if word in desc_lower])
    
    # Save it as a dictionary
    data_rows.append({
        'Code': code,
        'Description': description,
        'Routine_Score': r_score,
        'NonRoutine_Score': nr_score
    })
# 6. Convert the list of dictionaries into a pandas DataFrame called df_nco
df_nco = pd.DataFrame(data_rows)

# 7. Drop duplicate codes (a code might appear in a header and a footer)
df_nco = df_nco.drop_duplicates(subset=['Code'])

print(f"Built df_nco with {len(df_nco)} unique codes.")
print(df_nco[['Code', 'Routine_Score', 'NonRoutine_Score']].head())

Built df_nco with 183 unique codes.
    Code  Routine_Score  NonRoutine_Score
0   6111              0                 0
1   6115              0                 2
8   6112              0                 0
27  6113              0                 2
31  0206              0                 0


In [14]:
df_nco['Code_3D'] = df_nco['Code'].str[:3]

In [15]:
df_nco

,Code,Description,Routine_Score,NonRoutine_Score,Code_3D
0,6111,Field Crop and Vegetable Growers weeding and ...,0,0,611
1,6115,Medicinal and Aromatic Plant animals and nurs...,0,2,611
8,6112,Title Tree and Shrub Crop Growers ISCO 08 Uni...,0,0,611
27,6113,.0200 operations to cultivate and maintain Nur...,0,2,611
31,0206,installation of greenhouse structure as per Q...,0,0,020
...,...,...,...,...,...
1563,9611,Occupations in this Sub Division are Title Ga...,0,1,961
1567,9613,Title Sweepers and Related Labourers Family 9...,0,0,961
1571,9621,"Title Messengers, Package Deliverers 9621 Mes...",1,0,962
1572,9629,"Elementary Workers, Not any combination of fo...",0,0,962


In [16]:
df_nco['Routine_Intensity'] = df_nco['Routine_Score'] / (df_nco['Routine_Score'] + df_nco['NonRoutine_Score'] + 1)
#+1 in the denominator is to avoid not defined cases where the sum is 0 


In [17]:
exposure_index = df_nco.groupby('Code_3D').agg({
    'Routine_Score': 'mean',
    'NonRoutine_Score': 'mean',
    'Routine_Intensity': 'mean'
}).reset_index()

In [18]:
exposure_index

,Code_3D,Routine_Score,NonRoutine_Score,Routine_Intensity
0,020,0.500000,0.333333,0.250000
1,021,1.000000,0.000000,0.500000
2,030,0.000000,1.000000,0.000000
3,040,0.200000,0.000000,0.100000
4,041,0.666667,0.000000,0.222222
...,...,...,...,...
58,941,0.500000,0.000000,0.250000
59,951,0.000000,0.000000,0.000000
60,952,0.000000,0.000000,0.000000
61,961,0.000000,0.500000,0.000000


In [20]:
# 1. Load your clean PLFS dataset
plfs_path = "/Users/abhishek/Desktop/world-bank-indicators-project/clean_plfs_panel_2022_2024.csv"
plfs = pd.read_csv(plfs_path)
plfs['Occ_Code_Str'] = plfs['Occ_Code_Str'].astype(str).str.split('.').str[0]

# 2. Merge the exposure index onto the PLFS data
merged_df = plfs.merge(
    exposure_index, 
    left_on='Occ_Code_Str', 
    right_on='Code_3D', 
    how='left'
)

In [21]:
merged_df

,Year,State_UT_Code,Sector,Sex,Age,Principal_Status_Code,Principal_Industry_Code,Principal_Occupation_Code,CWS_Status_Code,CWS_Industry_Code,CWS_Occupation_Code,CWS_Earnings_Salaried,CWS_Earnings_SelfEmployed,Subsample_Multiplier,Occ_Code_Str,Code_3D,Routine_Score,NonRoutine_Score,Routine_Intensity
0,2022,28,1,1,52,11,1113.0,611.0,11,1.0,611.0,0,12000,1293611.0,611,611,0.000000,1.166667,0.000000
1,2022,28,1,1,18,31,84119.0,335.0,31,84.0,335.0,5000,0,1293611.0,335,NaN,NaN,NaN,NaN
2,2022,28,1,2,45,21,1113.0,611.0,21,1.0,611.0,0,0,1293611.0,611,611,0.000000,1.166667,0.000000
3,2022,28,1,2,22,31,84119.0,335.0,31,84.0,335.0,12000,0,1293611.0,335,NaN,NaN,NaN,NaN
4,2022,28,1,1,51,11,1113.0,611.0,11,1.0,611.0,0,15000,1293611.0,611,611,0.000000,1.166667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
821685,2024,34,2,2,28,92,NaN,NaN,92,NaN,NaN,0,0,1281861.0,nan,NaN,NaN,NaN,NaN
821686,2024,34,2,2,59,94,NaN,NaN,94,NaN,NaN,0,0,1281861.0,nan,NaN,NaN,NaN,NaN
821687,2024,34,2,1,24,31,47211.0,522.0,31,47.0,522.0,12000,0,961396.0,522,NaN,NaN,NaN,NaN
821688,2024,34,2,1,55,51,41001.0,711.0,51,41.0,711.0,0,0,737070.0,711,711,0.166667,0.500000,0.055556


In [22]:
matched = merged_df['Routine_Intensity'].notna().sum()
total = len(merged_df)
print(f"Matched exposure scores for {matched:,} out of {total:,} PLFS rows ({(matched/total)*100:.1f}%).")

Matched exposure scores for 303,069 out of 821,690 PLFS rows (36.9%).


In [23]:
# How many people in PLFS actually have an occupation code?
has_occupation = plfs['CWS_Occupation_Code'].notna().sum()
print(f"Rows with a job: {has_occupation:,}")
print(f"Rows without a job (NaN): {len(plfs) - has_occupation:,}")

Rows with a job: 454,033
Rows without a job (NaN): 367,657


In [24]:
# 1. Get the unique 3-digit codes from PLFS where the person has a job
plfs_codes = set(plfs['Occ_Code_Str'].dropna().unique())

# 2. Get the unique 3-digit codes from our PDF extraction
pdf_codes = set(exposure_index['Code_3D'].unique())

# 3. Find the codes in PLFS that are missing from the PDF
missing_codes = plfs_codes - pdf_codes

print(f"Total unique job codes in PLFS: {len(plfs_codes)}")
print(f"Total unique job codes in PDF: {len(pdf_codes)}")
print(f"\nWe are missing {len(missing_codes)} codes from the PDF.")
print("Here are the missing codes:", sorted(list(missing_codes)))

Total unique job codes in PLFS: 128
Total unique job codes in PDF: 63

We are missing 80 codes from the PDF.
Here are the missing codes: ['111', '112', '121', '122', '131', '132', '133', '134', '141', '142', '143', '211', '212', '213', '214', '215', '216', '221', '222', '223', '224', '225', '226', '231', '232', '233', '234', '235', '241', '242', '243', '251', '252', '261', '262', '263', '264', '265', '311', '312', '313', '314', '315', '321', '322', '323', '324', '325', '331', '332', '333', '334', '335', '341', '342', '343', '351', '352', '411', '412', '413', '421', '422', '431', '432', '441', '511', '512', '513', '514', '515', '516', '521', '522', '523', '524', '531', '532', '541', 'nan']


In [25]:
# 1. Create a list to hold the fallback rows
fallback_rows = []

# 2. Loop through the missing codes
for code in missing_codes:
    if code == 'nan':
        continue
        
    first_digit = code[0]
    
    # Assign default scores based on the 1-digit major group
    if first_digit in ['1', '2']:
        r_score, nr_score = 0, 3  # Managers/Professionals are highly non-routine
    elif first_digit in ['3', '5']:
        r_score, nr_score = 1, 2  # Technicians/Services are mixed
    elif first_digit in ['4', '6', '7', '8', '9']:
        r_score, nr_score = 3, 0  # Clerical/Manual are highly routine
        
    fallback_rows.append({
        'Code_3D': code,
        'Routine_Score': r_score,
        'NonRoutine_Score': nr_score,
        'Routine_Intensity': r_score / (r_score + nr_score + 1)
    })

# 3. Convert to dataframe and append to our exposure index
fallback_df = pd.DataFrame(fallback_rows)
final_exposure = pd.concat([exposure_index, fallback_df], ignore_index=True)

# 4. Re-run the merge with the fixed index
merged_df = plfs.merge(
    final_exposure, 
    left_on='Occ_Code_Str', 
    right_on='Code_3D', 
    how='left'
)

# 5. Check the match rate AGAIN, but only for people who actually have a job
employed = merged_df[merged_df['CWS_Occupation_Code'].notna()]
matched = employed['Routine_Intensity'].notna().sum()
print(f"Matched exposure scores for {matched:,} out of {len(employed):,} employed PLFS rows ({(matched/len(employed))*100:.1f}%).")

Matched exposure scores for 454,033 out of 454,033 employed PLFS rows (100.0%).


In [26]:
# Set random seed so you get the same random sample every time you run it
import random
random.seed(42)

# Get 10 random rows from the original df_nco (the table with the text)
sample = df_nco.sample(10, random_state=42)

for index, row in sample.iterrows():
    print(f"--- Code: {row['Code']} ---")
    print(f"Description: {row['Description'][:200]}...") # First 200 chars
    print(f"Score: Routine={row['Routine_Score']}, Non-Routine={row['NonRoutine_Score']}")
    print("\n")

--- Code: 6310 ---
Description:  Subsistence Crop Farmers and their households. Tasks performed by workers in this sub- Family 6310 major group usually include- preparing the Subsistence Crop Farmers soil; sowing, planting, tending ...
Score: Routine=0, Non-Routine=1


--- Code: 7213 ---
Description: .0301 prefabricated sheet metal sections according to specifications by various Dent Remover/Auto Body Repair mechanical processes such as drilling, Technician/Denter riveting, trimming, soldering etc...
Score: Routine=0, Non-Routine=1


--- Code: 9111 ---
Description:  Title Domestic Cleaners and Helpers Domestic, Hotel and Office Cleaners and Helpers sweep, vacuum clean, wash, 9111.0200 polish, take care of household linen, purchase household supplies; perform Cha...
Score: Routine=0, Non-Routine=0


--- Code: 8132 ---
Description:  equipment which makes photographic Title Photographic Products Machine film and paper, and which processes Operators exposed photographic film and makes prin

In [28]:
# 1. Add the new manual keywords to the routine list
routine_words = ['machine', 'operate', 'assemble', 'repeat', 'monitor', 'record', 
                 'calculate', 'check', 'sort', 'file', 'sweep', 'sow', 'plant', 
                 'drive', 'clean', 'dig', 'lift']

# 2. Re-score the df_nco
def score_exposure(text):
    text_lower = text.lower()
    r_score = sum([1 for word in routine_words if word in text_lower])
    nr_score = sum([1 for word in nonroutine_words if word in text_lower])
    return r_score, nr_score

df_nco[['Routine_Score', 'NonRoutine_Score']] = df_nco['Description'].apply(
    lambda x: pd.Series(score_exposure(x))
)

# 3. Re-calculate intensity
df_nco['Routine_Intensity'] = df_nco['Routine_Score'] / (df_nco['Routine_Score'] + df_nco['NonRoutine_Score'] + 1)

# 4. Re-aggregate to 3-digit
exposure_index = df_nco.groupby('Code_3D').agg({
    'Routine_Score': 'mean',
    'NonRoutine_Score': 'mean',
    'Routine_Intensity': 'mean'
}).reset_index()

# 5. Add back the 80 fallback codes
final_exposure = pd.concat([exposure_index, fallback_df], ignore_index=True)

# 6. Merge with PLFS and save
merged_df = plfs.merge(final_exposure, left_on='Occ_Code_Str', right_on='Code_3D', how='left')
merged_df.to_csv("/Users/abhishek/Desktop/world-bank-indicators-project/final_plfs_with_exposure.csv", index=False)
print("Final file saved!.")

Final file saved!.
